In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# Load data
data = pd.read_csv('../data/featured_system_metrics.csv')

# Transform
data['net_bytes_per_sec'] = np.log1p(data['net_bytes_per_sec'])
data["net_change"] = np.sign(data["net_change"]) * np.log1p(np.abs(data["net_change"]))

data = data.dropna()

# Split
train_size = int(len(data) * 0.8)
train = data[:train_size]
test = data[train_size:]

# Targets
targets = ["cpu_percent", "ram_percent", "net_bytes_per_sec"]

# Remove timestamp
X_train = train.drop(columns=['timestamp'])
X_test = test.drop(columns=['timestamp'])

# Correct cols_to_scale
cols_to_scale = [
    col for col in X_train.columns 
    if col not in targets
]

# Scaling
scaler = MinMaxScaler()

X_train_scaled_part = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled_part = scaler.transform(X_test[cols_to_scale])

joblib.dump(scaler, "../data/scaler.pkl")
joblib.dump(cols_to_scale, "../data/scale_columns.pkl")

# Convert scaled part
X_train_scaled_part = pd.DataFrame(X_train_scaled_part, columns=cols_to_scale)
X_test_scaled_part = pd.DataFrame(X_test_scaled_part, columns=cols_to_scale)

# Combine back with targets
train_final = pd.concat([ train[targets].reset_index(drop=True), X_train_scaled_part], axis=1)
test_final = pd.concat([test[targets].reset_index(drop=True), X_test_scaled_part], axis=1)

# Save final data
train_final.to_csv("../data/train_data.csv", index=False)
test_final.to_csv("../data/test_data.csv", index=False)